In [1]:
import os
import ROOT as root
import numpy as np
from array import array
import glob
import re
import pandas as pd

file = root.TFile("/Users/icosivi/cernbox/MTD/QAQC/root_files/HPK5_16x16_IVtree.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
pad = array('i', [0])
sensor = array('i', [0])

V = root.std.vector("float")()
IPAD = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("pad", pad, 'pad/I')
tree.Branch("sensor", sensor, 'sensor/I')


V.reserve(1000)
IPAD.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("IPAD", "std::vector<float>", IPAD)

txt_files = glob.glob("/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx")

df_3 = pd.read_excel('/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx', sheet_name='W No.3', engine='openpyxl', header=None)
df_4 = pd.read_excel('/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx', sheet_name='W No.4', engine='openpyxl', header=None)
df_6 = pd.read_excel('/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx', sheet_name='W No.6', engine='openpyxl', header=None)
df_7 = pd.read_excel('/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx', sheet_name='W No.7', engine='openpyxl', header=None)
df_8 = pd.read_excel('/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx', sheet_name='W No.8', engine='openpyxl', header=None)
df_9 = pd.read_excel('/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx', sheet_name='W No.9', engine='openpyxl', header=None)
df_10 = pd.read_excel('/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx', sheet_name='W No.10', engine='openpyxl', header=None)
df_11 = pd.read_excel('/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx', sheet_name='W No.11', engine='openpyxl', header=None)
df_12 = pd.read_excel('/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx', sheet_name='W No.12', engine='openpyxl', header=None)
df_28 = pd.read_excel('/Users/icosivi/cernbox/MTD/QAQC/HPKdata/HPK5/20250109_Id data.xlsx', sheet_name='W No.28', engine='openpyxl', header=None)

dataframes = {
3: df_3.iloc[2:23,1:],
4: df_4.iloc[2:23,1:],
6: df_6.iloc[2:23,1:],
7: df_7.iloc[2:23,1:],
8: df_8.iloc[2:23,1:],
9: df_9.iloc[2:23,1:],
10: df_10.iloc[2:23,1:],
11: df_11.iloc[2:23,1:],
12: df_12.iloc[2:23,1:],
28: df_28.iloc[2:23,1:]
}

counter = 0
pad_counter = 0
sensor_counter = 0

for number, df in dataframes.items():
 
 counter = 0
 pad_counter = 0
 sensor_counter = 0
 V.clear()
 IPAD.clear()
 
 for (columnName, columnData) in df.items():
  if( float(columnName)!=1 ):  
    pad_counter += 1 
    V.clear()
    IPAD.clear()
    event[0] = counter
    wafer[0] = int(number)
    pad[0] = pad_counter
    sensor[0] = sensor_counter
    
    if pad_counter%256 == 0:
      pad_counter = 0
      sensor_counter += 1
      
    for idx, col in enumerate(columnData):
      if not pd.isna(col):
       IPAD.push_back( float(col) )
       V.push_back( float(df.iloc[idx,0]) )
			
    tree.Fill()
    counter += 1

tree.Write()
file.Write()
file.Close()

Welcome to JupyROOT 6.30/04


In [ ]:
import os
import ROOT as root
import numpy as np
from array import array
import glob
import re

file = root.TFile("/Users/icosivi/cernbox/MTD/QAQC/root_files/HPK5_16x16_IVtree.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
type = array('i', [0])
row = array('i', [0])
column = array('i', [0])

V = root.std.vector("float")()
IBACK = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("type", type,'type/I')
tree.Branch("row", row, 'row/I')
tree.Branch("column", column, 'column/I')


V.reserve(1000)
IBACK.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("IBACK", "std::vector<float>", IBACK)

txt_files = glob.glob("/Users/icosivi/cernbox/MTD/QAQC/FBKdata/UFSD_LF/IV_LC1/DEV_16x16/*.txt")

counter = 0

for txt in txt_files:
    #print(txt)
    V.clear()
    #IBACK.clear()

    with open(txt,"r") as t:
        lines_list = t.readlines()
        header_v = lines_list[0]
        
        header_v.strip()
        hv = re.split("\s+", header_v)
        
        for m in hv[6:]:
            if not m.isspace():
                if m:
                    #print(m)
                    V.push_back( abs(float(m)) )

        lines = lines_list[1:]
        
        for line in lines:
            line.strip()
            IBACK.clear()
            ll = re.split( '\s+', line)
            #print(ll[4])
            if ll[4] == "I_BACK":
                type_def = re.split("_", ll[3])
                #print(type_def[0])
                if(type_def[2]=='PIN'):
                  type[0] = 0
                  event[0] = counter
                  wafer[0] = int( ll[0] )
                  column[0] = int( ll[1] )
                  row[0] = int( ll[2] )
                elif(type_def[2]=='PAD'):
                  type[0] = 1
                  event[0] = counter
                  wafer[0] = int( ll[0] )
                  column[0] = int( ll[1] )
                  row[0] = int( ll[2] )
                
                '''
                sensor_types = re.split( '_|-', ll[3])
                ggtype = re.findall(r'\d+',sensor_types[1])
                gr_type[0] = int(ggtype[0])
                for s in sensor_types[1]:
                    if s.isdigit():    
                        gr_type[0] = int( s )
                
                for p in sensor_types[2]:
                    if p.isdigit():    
                        grn[0] = int( p )
                for z in sensor_types[2]:
                    if z.isdigit():    
                        grt[0] = int( z )
                    if z == "S":
                        grt[0] = int(2)
                '''
                for q in ll[6:]:
                    if not q.isspace():
                        if q:
                            IBACK.push_back( abs(float(q)) )
                            #if counter==2:
                                #print(q)          
                #if not any(x == "PIN" for x in sensor_types):
                tree.Fill()
                counter += 1

tree.Write()
file.Write()
file.Close()